# Importación

In [ ]:
import os
import re
import pandas as pd

## Carpeta Local

In [ ]:
folder = "cuestionarios"

## Carpeta en Drive (con Colab)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Obtener todos los xlsx de la carpeta compartida
folder = "/content/drive/MyDrive/Cuestionarios/xlsx"

## Proceso de Importación

In [ ]:
files = os.listdir(folder)
excel_files = sorted([f for f in files if f.endswith('.xlsx')])

# Guardar todos los datos como dataframes, dentro de una lista de dataframes
questionnaires = []
filenames = []

for file in excel_files:
  file_path = os.path.join(folder, file) #Busca los archivos en la carpeta
  file_df = pd.read_excel(file_path)     #Lee los datos de los archivos
  # strip() sobre los nombres de columnas
  file_df.columns = file_df.columns.str.strip()  #Definimos los encabezados
  questionnaires.append(file_df)                 #Crea un arreglo
  filenames.append(os.path.basename(file_path))  #

# Guía para saber qué número corresponde a qué archivo
for filename in filenames:
  print(f"{filenames.index(filename)} es {filename}")

# Limpieza

In [ ]:
def clean_text(text):
    # Todo a minúsculas
    text = text.lower()
    # Quitar caracteres no ASCII
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    # Quitar signos de puntuación, a excepción de .,:;
    text = re.sub(r'[^\w\s.,:;]', '', text)
    # Normalizar espacios en blanco
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Lista de DataFrames donde se almacenarán los datos limpios
questionnaires_clean = []
for questionnaire in questionnaires:
  # Mismo cuestionario con mismas columnas
  questionnaires_clean.append(questionnaire.iloc[0:0])

# Aplicar la función de limpieza a cada respuesta en los cuestionarios
for questionnaire_i, questionnaire in enumerate(questionnaires):
    for column in questionnaire.columns:
        # Saltar las columnas de "marca temporal"
        if "marca temporal" not in column.lower():
            questionnaires_clean[questionnaire_i][column] = questionnaire[column].astype(str).apply(clean_text)
        else:
            questionnaires_clean[questionnaire_i][column] = questionnaire[column]

# Clasificación

## Función de Clasificación

In [ ]:
# Función que clasifica una respuesta dada un mapeo palabras -> categoría
def classify_answer(answer, mapping):

  # ¿Qué es esto y para qué está?
  """
  # Precompilamos las expresiones regulares para cada categoría
  compiled_patterns = {
    classification: [re.compile(pattern, re.IGNORECASE) for pattern in patterns]
    for classification, patterns in mapping.items()
  }
  """

  # Si la respuesta es nula
  if pd.isna(answer):
    return "nulo"

  for classification, patterns in mapping.items():
    for pattern in patterns:
      if re.search(pattern, answer, re.IGNORECASE):
        return classification
  # unclassified
  return None

## Mappings

(es decir, clasificaciones que aparecen en varias preguntas en varios cuestionarios)

In [415]:
# Mapeo para clasificación
# clasificación: [lista de palabras a buscar en la respuesta para clasificar]
# (palabras son en realidad expresiones regulares)

mapping_intensity = {
    "nada": ["nada", "no afecta", "ninguno", "no impacta", "ninguna"],
    "muy poco": ["poco", "no tanto", "deficiente", "puede mejorar", "sigo trabajando", "miedo"],
    "poco": ["algo", "masomenos", "medianamente", "regular", "me defiendo", "intermedio", "aveces bien", "a veces"],
    "moderado": ["suficiente", "bien", "preparad", "la mayoría de veces bien", "c.mod"],
    "mucho": ["mucho", "bastante", "bastante preparado"],
    "muy alto": ["super bien","demasiado", "considerablemente", "gran", "m.s", "alta", "completamente", "muy"],
    "desconocido": ["no s", "no lo sé", "ya veremos", "depende"]
}

mapping_yes_no = {
    "sí": ["sí", "si"],
    "no": ["no"]
}

mapping_gender = {
    "masculino": ["masculino", "hombre"],
    "femenino": ["femenino", "mujer"],
    "otro": ["no binario", ".+"]
}

mapping_soft_skills = {
    "comunicación": ["comunica", "explic", "expres", "ponencia", "habl","expo"],
    "trabajo en equipo": ["equipo", "cooper", "integr", " dem.s"],
    "liderazgo": ["lider"],
    "paciencia": ["paciencia"],
    "pensamiento creativo/crítico": ["pensamiento creativo", "pensamiento cr.tico"],
    "habilidades sociales": ["relaci", "sociales"],
    "ventas y persuasión": ["vend", "persua"],
    "visión interdisciplinaria": [".reas", "fuera"],
    "manejo de emociones": ["emociones"],
    "valentía": ["valentía"],
    "design thinking": ["design thinking"],
    "todas": ["todas", "complemento"],
    "ninguna": ["no", "ninguna", "de que habilidades hablamos"]
}

mapping_materias = {
    "computacional": [
        "Computacional", "Computación", "Desarrollo de software", "Desarrollo de software o Ciberseguridad",
        "Seguridad Computacional", "Seguridad Computacional o bases de datos", "Bases de datos",
        "Datos y desarrollo web",  "Datos o Seguridad Computacional",
        "bases de datos", "Programación Orientada a Objetos", "POO", "Estructuras de datos", "Bases de datos",
        "Administración de bases de datos","Ingeniería de datos","Seguridad Computacional", "desarrollo web",
        "POO", "Programación",
        "Redes de computo",  "Bases de datos",
        "Metodos numéricos", "teoría de grafos", "Programación Orientada a Objetos",  "Redes de computadoras",
          "Computación", "Seguridad Computacional", "modelado matemático", "Desarrollo de software",
        "Seguridad Computacional o bases de datos", "Redes de computo", "Redes de computadoras",
        "Seguridad Computacional", "Temas selectos de computación"
    ],
    "matemáticas": [
        "matemática", "matemáticas computacionales", "matemáticas", "teoría de grafos",
        "probabilidad y estadística", "Cálculo", "Optimización", "Optimización II","matemáticas discretas",
        "Estadística", "Probabilidad", "Análisis de datos",
         "modelado matemático", "Estadística 1", "Estadística 2", "Optimización", "Cálculo",
        "Ecuaciones diferenciales","Algoritmos",
        "Álgebra lineal", "Cálculo"
    ],

    "datos": [
        "Análisis de datos", "Datos", "Ciencia de datos",
        "Ciencia de datos", "Científico de datos", "Datos", "Análisis de datos",
        "Ingeniería de datos",  "Análisis de datos", "Ciencia de datos", "Científico de datos", "Datos y desarrollo web", "Análisis de datos",
        "Ingeniería de datos", "Estadística", "Estadística y probabilidad", "modelado matemático", "Análisis de datos"
    ],

    "otras": [
         "Finanzas",  "Inglés", "Seminarios"
    ]
}

mapping_software = {
    "linux": [
        "Linux", "Linux", "Linux", "linux"
    ],
    "bases de datos": [
        "SPSS", "SQL Workbench", "MySQL Workbench", "SAS", "SQL", "SQL Workbench"
    ],
    "lenguaje de programacion": [
         "wift", "R", "c, c++", "C", "C++", "ava", "ython"
    ],
    "entorno de programacion": [
        "isual Studio", "VS Code", "isual studio", "etbeans", "Visual Studio Code",
        "olfram Mathematica", "olfram",
    ],
    "hardware": [
        "RAM", "ram", "Cisco"
    ],
    "otro": [
        "Paquete", "github", "brave", "El de optimización", "odos", "inguno",
        "Latex jaja", "No se", "Ingl.s", "Paqueter.a de Office", "aquinas", "no "
    ]
}

mapping_raiz_debido_a_materias = {
    "no debe": [
        "no debo", "No debo", "No debo", "no debo", "no", "o hay", "ninguna"
    ],
    "formas de evaluacion": [
        "valuaci.n"
    ],
    "falta de interes o dificultades": [
        "lojera", "esinterés", "enseñanza", "ificultades", "eprobado",  "culpa", "ensa",
    ],
    "pandemia": [
        "andemia", "alud"
    ],
    "problemas de tiempo": [
        "tiempo", "vida",  "poco", "Tiempo", "imult.neamente",
        " full time", "istancia"
    ],
    "problemas economicos y laborales": [
        "rabajar","rabajo", "aborales", "Laborales", "tener trabajo", "trabajo", "El trabajo", "Laborales",
        "econom.a"
    ],

    "razones personales": [
        "mpezar ", "ulpa", "nfermedad ", "mpe.o", "epresión",
        "econ.mico", "emocional", "amigo",
        "amiliares", "Sociales", "novi.s", "fiestas", "ociales", "ntrapersonales"
    ],
    "indefinido": [
        "No lo sé"
    ]
}

mapping_cambios = {
    "mejoras academicas": [
        "mejores planes de estudio", "evaluación",
        "temarios", "profesores","técnicas pedagógicas",
        "enseñanza", "seríacion",
        "regularización", "clases", "diapositivas"
    ],
    "flexibilidad y tiempo": [
        "flexibilidad",  "planificar", "libre",
        "horario", "tiempo ", "separar la clase en dos horarios",
        "horarios más diversos", "línea", "laboratorio","híbrido"
    ],
    "salud mental y bienestar": [
        "menos carga", "alud mental", "descansos", "descansado",
        "auto calificables", "interactiva"
    ],
    "actitud y enfoque": [
        "ayuda", "concentrarse",
        "teórica", "preparados", "actitud", "positiva"
    ],
    "otros": [
        "problema no es el sistema", "ninguna",
         "viernes"
    ]
}


## Tipos de Preguntas
(para clasificar todas las respuestas de todos los cuestionarios)

In [417]:
# Una lista de diccionarios de tipos de pregunta cada cuestionario
# pregunta: tipo
question_type = [{} for questionnaire in questionnaires]
# Estos tipos pueden ser strings (trivial entenderlos) u otro diccionario
# Si son un diccionario, se trata de un mapeo para clasificación
# clasificación: [lista de palabras a buscar en la respuesta para clasificar]
# (palabras son en realidad expresiones regulares)

# 0 – CaminoProfesionalMAC1.xlsx
question_type[0] = [
    # Marca temporal
    "datetime",
    # Número de cuenta
    "account_number",
    # ¿Cuál es tu mayor preocupación al terminar tus estudios?
    {
        "empleo": [
            "no encontrar trabajo", "quedarme sin casa", "buen empleo", "sin empleo",
            "conseguir chamba", "trabajo", "encontrar trabajo"
        ],
        "crecimiento personal": [
            "qué hacer con mi vida", "felicidad", "rumbo de la sociedad"
        ],
        "finalizacion estudios": [
            "titularme"
        ],
        "condiciones laborales": [
            "jubilarme", "no me haya gustado la carrera"
        ],
        "habilidades y preparacion": [
            "nivel de inglés"
        ],
        "sin preocupaciones": [
            "ninguna", "hasta ahorita ninguna"
        ]
    },
    # ¿Qué factores influyen más en tu decisión sobre qué camino seguir después de egresar?
    {
        "economicos": [
            "dinero", "sueldo", "salario", "remuner", "subsistir", "comer"
        ],
        "familiares y ubicacion": [
            "familia", "cerca de mi casa", "ubicaci", "traslado"
        ],
        "desarrollo personal y felicidad": [
            "felicidad", "me hace realmente feliz", "gustos", "preferencias personales"
        ],
        "oportunidades y crecimiento": [
            "oportunidades", "crecimiento", "proyectos", "domin", "conocimientos"
        ],
        "sociales y emocionales": [
            "sociales", "emocionales", "interpersonales", "políticos", "naturales", "confianza"
        ],
        "trabajo actual": [
            "chamba", "trabajo actual"
        ],
        "vocacion y estudios": [
            "materias", "curs", "saber que es lo que voy a hacer el resto de mi vida"
        ]
    },
    # ¿Cuál es tu situación actual respecto a la elección de un camino profesional después de egresar?
    {
        "camino definido": [
            "tengo un camino pensado", "ya estoy trabajando", "cuento con trabajo",
            "especialista", "claro", "empezado", "me gusta"
        ],
        "indecision o confusion": [
            "indeciso", "confundida", "no he elegido", "indefinida", "complicado elegir", "posponiendo"
        ],
        "busqueda de trabajo": [
            "buscando", "encontrar un puesto", "buenas opciones de trabajo"
        ],
        "educacion continua": [
            "maestría", "especialización", "titulo"
        ],
        "tranquilidad o sin preocupaciones": [
            "tranquila", "no me da miedo", "chida", "cómodo"
        ]
    },
    # ¿Qué tipo de apoyo crees que te ayudaría más en esta etapa de decisión?
    {
        "transporte": [
            "transporte"
        ],
        "experiencia practica": [
            "prácticas", "rotación", "inducción profesional"
        ],
        "apoyo economico": [
            "económico", "1000000", "pesos"
        ],
        "orientacion profesional": [
            "orientación", "conferencias", "pláticas", "guía", "egresados",
            "ferias", "conocer a personas", "alguien de fuera", "hablar con los profesores"
        ],
        "capacitación": [
            "cursos", "sectores específicos", "programa para encontrar trabajos"
        ],
        "apoyo psicologico": [
            "psicologico"
        ],
        "ninguno o indeciso": [
            "ninguno", "nain", "no sé"
        ]
    },
    # ¿Qué habilidades o conocimientos crees que son esenciales para tomar una decisión sobre tu especialización o carrera profesional después de egresar?
    {
        "conocimiento del campo laboral": [
            "campo laboral", "ofertas de vacantes", "futuro laboral", "sueldos",
            "perfiles de las ofertas", "actividades a realizar"
        ],
        "autoconocimiento y toma de decisiones": [
            "autoconocimiento", "saber qué quieres", "qué te gusta", "para qué eres bueno",
            "decisión acertada", "panorama completo"
        ],
        "habilidades tecnicas": [
            "programación", "estadística", "computación", "mates aplicadas"
        ],
        "habilidades blandas": [
            "pensamiento crítico", "comunicación", "dominio del inglés", "expresar lo que quiero"
        ],
        "networking": [
            "networking", "conocer personas", "red de contactos"
        ],
        "preparacion para el empleo": [
            "entrevistas", "creación de cv", "considerado en vacantes"
        ]
    },
    # ¿En qué medida consideras que las prácticas profesionales o proyectos extracurriculares influirán en tu decisión de que camino seguir?
    {
        "alta influencia": [
            "bastante", "influyen mucho", "demasiado", "mucho",
            "gran impacto", "mucha", "en la más alta medida"
        ],
        "media influencia": [
            "algo", "descubrir que me ayuda más", "gracias a ellas sé a dónde no quiero ir"
        ],
        "baja influencia": [
            "poco", "he hecho varios proyectos", "ninguna", "no refleja la vida laboral",
            "en ninguno", "nada"
        ],
        "inverso": [
            "el camino que elija influirá en las prácticas", "no al revés"
        ]
    },
    # ¿Cómo visualizas tu carrera profesional dentro de 5 años? ¿Qué tipo de trabajo o proyectos te gustaría estar realizando?
    mapping_materias,
    # ¿Qué es lo que más valoras en un posible empleo o área de especialización después de terminar tu carrera?
    {
        "salario": [
            "salario", "sueldo", "relación sueldo-carga de trabajo"
        ],
        "ambiente laboral": [
            "ambiente laboral", "flexible", "buenos horarios"
        ],
        "aprendizaje y crecimiento": [
            "seguir aprendiendo", "nuevos retos", "enseñar habilidades",
            "oportunidad de crecimiento", "posibilidad de escalar"
        ],
        "intereses personales": [
            "intereses", "hacer algo que me apasione", "valga más como persona",
            "gustos", "sumen", "proyectos"
        ],
        "seguridad empleo": [
            "empleo"
        ]
    },
    # ¿Qué tan útil consideras el material de estudio de la carrera para conseguir trabajo?
    mapping_intensity,
    # ¿Sientes que tu carrera universitaria te permitió desarrollar una red de contactos profesionales que te ha ayudado en tu camino laboral?
    mapping_yes_no,
    # ¿Qué aspectos de tu formación académica crees que podrían mejorarse para preparar mejor a los futuros egresados?
    {
        "prácticas profesionales": [
            "prácticas profesionales", "internships", "contacto con lo que se usa en un trabajo real",
            "acercamiento más temprano al ámbito laboral"
        ],
        "habilidades blandas": [
            "habilidades blandas", "IBM"
        ],
        "vinculación empresarial": [
            "relación laboral", "empresas", "red de contactos", "comunidad"
        ],
        "actualización plan estudios": [
            "actualización", "mejorar los temarios", "plan de estudios", "optativas",
            "temarios", "laboratorios"
        ],
        "aprendizaje práctico": [
            "más práctico", "experiencias cercanas a lo profesional", "ejemplos de la vida laboral"
        ],
        "flexibilidad horaria": [
            "oportunidades en los horarios", "clases en línea", "poder trabajar y estudiar"
        ]
    },
    # ¿Consideras que el contenido de tu carrera estuvo actualizado con las necesidades actuales del mercado laboral?
    mapping_yes_no
]

# 1 – Finanzas Transparentes (Respuestas).xlsx
question_type[1] = [
    # Marca temporal
    "datetime",
    # Número de cuenta
    "account_number",
    # 1.  ¿Cuánto dinero generas al mes?
    "numeric",
    # 2. ¿Cuánto gastas mensualmente considerando todos tus gastos?
    "numeric",
    # 3. ¿Cuánto dinero logras ahorrar al mes?
    "numeric",
    # 4. ¿Tienes dinero reservado para emergencias? Si sí, ¿cuánto aproximadamente?
    "numeric",
    # 5. ¿Tu dinero tiene liquidez o está comprometido en inversiones o deudas?
    {
		"liquidez": ["liquidez", "sí", "si"],
		"inversiones o deudas": ["inversiones", "deudas"],
		"desconocido": ["no sé", "no"]
    },
    # 6. ¿Cuentas con inversiones a corto plazo?
    mapping_yes_no,
    # 7. ¿Cuentas con inversiones a largo plazo?
    mapping_yes_no,
    # 8. ¿Tienes un plan personal de retiro o planeas tener uno en el futuro?
    {
        "no": ["no", "no tengo", "no planeo"],
		"sí": ["sí", "si", "planeo"]
    }
    # 9. ¿Cuántas tarjetas de crédito tienes?
    "integer",
    # 10. ¿Qué porcentaje de tu línea de crédito tienes disponible actualmente?
    "percentage",
]

# 2 – Habilidades de MAC (respuestas).xlsx
question_type[2] = [
    # Marca temporal
    "datetime",
    # Dirección de correo electrónico
    "email",
    # 1. ¿Cuál es tu numero de cuenta?
    "account_number",
    # 2. ¿Cómo describirías tu capacidad para explicar ideas complejas a personas que no son expertas en tu área?
    mapping_intensity,
    # 3. ¿Qué tan cómodo te sientes trabajando en equipo?
    mapping_intensity,
    # 4. ¿Cuál ha sido tu mayor desafío al trabajar en equipo y cómo lo superaste?
    {
        "comunicación": ["comunicaci.n", "explicar", "hablar", "efectiva"],
        "diferencias de ideas": ["diferencia de ideas", "negociar", "otras formas de hacer las cosas"],
        "falta de compromiso": ["abandono", "trabaj.. solo", "no trabajan", "apat.a", "hacer todo yo solo"],
        "organización": ["organizaci.n", "cronograma", "tareas", "calendario"],
        "colaboración con personas difíciles": ["personas complicadas", "no coopera", "no me agrada", "forzarlo"],
        "habilidades blandas": ["habilidades blandas", "adapt.ndome", "confianza", "l.mites"],
        "trato y actitud de compañeros": ["grosero", "compa.eros", "maltrato"]
    },
    # 5.¿Sientes que la carrera de MAC te ha preparado adecuadamente para el trabajo en equipo?
    mapping_yes_no,
    # 6.  ¿Qué tan fácil o difícil te resulta establecer nuevas conexiones?
    mapping_intensity,
    # 7. ¿Cómo te sientes al tener que hablar frente a una gran audiencia?
    {
        "miedo o ansiedad": ["nervios", "nervioso", "nerviosa", "ansioso", "intimidad.", "mal", "no me gusta"],
        "seguridad": ["preparaci.n", "m.s"],
        "inseguridad inicial pero mejora": ["al principio", "inicio", "luego agarro vuelo", "despu.s mejora"],
        "confianza": ["bien", "perfectamente", "c.modo"]
    },
    # 8. ¿Qué técnicas usas para manejar el estrés y mantener la calma en situaciones difíciles?
    {
        "sin técnicas": ["no tengo", "ninguna", "no conozco"],
        "respiración y relajación": ["respirar", "respiraci.n", "respiro", "cierro los ojos", "meditaci.n"],
        "ejercicio físico": ["ejercicio", "caminar", "salir "],
        "terapia o apoyo psicológico": ["terapia", "dtpa"],
        "autosugestión o pensamientos positivos": ["repetirme muchas veces", "recordar que eso no me va a matar"],
        "hábitos nerviosos": ["me como las u.as", "comiendo por ansiedad"],
        "descanso o pausas": ["break", "relajarme"]
    },
    # 9.  ¿Cómo te preparas para presentaciones orales o exposiciones frente a un público amplio?
    {
        "ensayo y práctica": ["practic", "ensay", "expon", "grab"],
        "estudio e investigación": ["investig", "estudi", "entender"],
        "elaboración de material": ["hacer la presentaci.n", "crear la presentaci.n", "preparo mi tema", "armar un gui.n"],
        "uso de estrategias narrativas": ["storytelling", "enamoro del tema"],
        "memorización de puntos clave": ["memoriza", "prepar"],
        "manejo de la ansiedad": ["tranquiliz", "personas"],
        "no preparación": ["no "]
    },
    # 10. ¿Crees que las habilidades técnicas (programación, matemáticas, algoritmos) son suficientes para desempeñarte en el ámbito laboral sin necesidad de habilidades blandas?
    mapping_yes_no,
    # 11. ¿Has aplicado alguna de las habilidades blandas aprendidas en estos cursos en tu vida académica o personal?
    mapping_yes_no,
    # 12. ¿Cuáles consideras que son las habilidades blandas más importantes para un profesional de Matemáticas Aplicadas y Computación?
    mapping_soft_skills,
    # 13.  ¿Cuáles de las siguientes habilidades blandas crees que te faltan desarrollar más
    mapping_soft_skills,
    # 14.  ¿Qué tan preparado/a te sientes en términos de habilidades blandas para enfrentar un empleo después de graduarte?
    mapping_intensity,
    # 15.  ¿Consideras que los estudiantes de MAC suelen subestimar la importancia de las habilidades blandas?
    mapping_yes_no
]

# 3 – Impacto del Horario en el Desempeño y Bienestar de los Estudiantes de Minería de Datos (respuestas).xlsx
question_type[3] = [
    # Marca temporal
    "datetime",
    # Dirección de correo electrónico
    "email",
    # Número de cuenta
    "account_number",
    # ¿Qué tanto afecta tener una materia de 18:00 a 20:00 hrs en viernes a tu desempeño académico?
    mapping_intensity,
    # ¿Qué tanto afecta tener una materia de 18:00 a 20:00 hrs en viernes a tu desempeño personal?
    mapping_intensity,
    # ¿Qué dificultades enfrentas para asistir y concentrarte en esta materia en ese horario? (Separa cada una por comas)
    "list",
    # ¿Qué tanto impacta este horario en tu organización del tiempo para otras actividades académicas o laborales?
    mapping_intensity,
    # ¿Crees que el rendimiento en una materia como Minería de Datos se ve afectado por el horario? ¿Por qué?
    mapping_yes_no,
    # ¿Qué tanto afecta este horario a tu alimentación y descanso?
    mapping_intensity,
    # ¿Qué sugerencias darías para mejorar la experiencia de aprendizaje en esta materia considerando el horario? (Menciona las sugerencias separando por comas)
    mapping_cambios,
    # ¿Has considerado dejar esta materia debido al horario? ¿Por qué?
    mapping_yes_no,
    # ¿Qué herramientas o métodos crees que podrían hacer más llevadera la clase en este horario? (Menciona las herramientas separando por comas)
    "list",
    # ¿Cuál es tu principal medio de transporte para llegar y salir de la universidad?
    {
        "auto": ["auto", "coche"],
        "público": ["camión", "combi", "público", "metro", "FES directo"],
        "caminar": ["camin"]
    },
    # ¿Has experimentado dificultades con el transporte debido al horario de la materia? Si es así, ¿cuáles?
    mapping_yes_no,
    # ¿Cuánto tiempo tardas en llegar a casa después de la clase? (Escribe tu respuesta en minutos)
    "quantity",
    # ¿Te sientes seguro al transportarse después de las 20:00 hrs? Explica tu respuesta.
    mapping_yes_no,
    # ¿Has tenido que modificar tus rutas o modos de transporte por la hora en que termina la clase?"
    mapping_yes_no
]

# 4 – Perfil Académico (respuestas).xlsx
question_type[4] = [
    # Marca temporal
    "datetime",
    # 1. Número de Cuenta:
    "account_number",
    # 2. Edad:
    "quantity",
    # 3. Genero
    mapping_gender,
    # 4. Estado donde resides:
    {
        "ciudad de méxico": ["CDMX", "Ciudad de M.xico"],
        "estado de méxico": ["Estad. de M.xico", "Edo Mex"],
        "guerrero": ["Guerrero"]
    },
    # 5. ¿Actualmente estas laborando o realizando tu Servicio Social?
    {
        "ninguno": ["ninguno", "no", "nain"],
        "servicio social": ["servicio"],
        "trabajo": ["trabaj", "labor"],
        "ambos": ["amb.s", "dos", "si", "sí"]
    },
    # 6. ¿Cuánto tiempo en minutos dura tu recorrido a la Facultad?
    "quantity",
    # 7.  ¿Que materias de la carrera te interesaron más las de área computacional o matemática?
    mapping_materias,
    # 8. ¿Tienes Automóvil?
    mapping_yes_no,
    # 9. ¿En qué Área de especialidad de la carrera estás interesado?
    mapping_materias,
    # 10. ¿Qué materia se te dificultó más en la carrera?
    mapping_materias,
    # 11. ¿Cuales son las 3 materias que consideras que más te aportado más en tu elección de área de especialidad?
    mapping_materias,
    # 12. ¿En cuantas y cuales materias elegiste a un profesor por "barquear" la materia?
    mapping_materias,
    # 13. ¿Te consideras bueno trabajando en equipo? ¿por qué?
    mapping_yes_no,
    # 14. ¿Te consideras perfeccionista?
    mapping_yes_no,
    # 15. ¿Tus padres o alguna persona cercana se encuentra laborando en algo relacionado con el área de especialidad que elegiste?
    mapping_yes_no,
    # 16. ¿Te gustaría crecer en una empresa siempre con el mismo proyecto durante años o preferirias trabajar en varios proyectos de diferentes temas?
    {
        "preferencia varios proyectos": [
            "arios ",
            "iferente"
        ],
        "preferencia un solo proyecto": [
            "ismo",
            "olo",
        ],
        "indeterminado": [
            "Si",
            "dos",
            "ambos",
            "No se",
            "epende"
        ]
    },
    # 17. ¿Qué persona consideras es tu modelo a seguir?
    {
        "familia": [
            "adres", "tíos", "abuela", "mam", "padre", "papas", "Mi mamá", "papá", "primo", "mi",
            "Mi", "mis", "Mis"
        ],
        "personas especificas": [
            "Héctor", "Bach", "maestros"
        ],
        "sin respuesta o indeterminada": [
            "nadie", "Nadie", "Aún no se", "No se"
        ],
        "variadas": [
            "varias"
        ]
    },
    # 18. ¿Consideras que tu equipo de computo es adecuado para el software requerido en la carrera?
    mapping_yes_no,
    # 19. ¿Qué software utilizado en la carrera consideras el mas complejo de usar?
    mapping_software,
    # 20. ¿Qué software utilizado en la carrera consideras el mas fácil de usar?
    mapping_software,
]

# 5 – Rendimiento académico (respuestas).xlsx
question_type[5] = [
    # Marca temporal
    "datetime",
    # ¿Cuál es tu promedio actual?
    "quantity",
    # Numero de cuenta
    "account_number",
    # ¿Cuántas horas dedicas al estudio semanalmente y cómo las distribuyes?
    "quantity",
    # ¿Cuántas materias debes actualmente?
    "quantity",
    # ¿Cuántas materias has reprobado durante la carrera?
    "quantity",
    # ¿Cuántos años llevas en la carrera?
    "quantity",
    # ¿Cuál es la raíz de que debas materias?
    mapping_raiz_debido_a_materias,
    # ¿Los profesores te han apoyado durante la carrera?
    mapping_yes_no,
    # ¿Qué factores externos (familiares, laborales, sociales) afectan tu desempeño académico?
    mapping_materias,
    # ¿Qué cambios o mejoras sugerirías en el sistema educativo para mejorar el rendimiento académico?
    mapping_cambios,
    # ¿Te parece que los exámenes reflejan de manera justa tu nivel de conocimiento? ¿Por qué?
    mapping_yes_no,
    # ¿Cómo gestionas tu tiempo entre tus responsabilidades académicas y otras actividades?
    {
        "priorizacion": [
            "prioridad",
            "Priorizar",
            "priorizo",
            "le echo ganas",
        ],
        "estructura y planificacion": [
            "cada una",
            "horario",
            "dividiendo", "organizándome"
            "limitar", "tiempo",
            "hrs"
        ],
        "equilibrio trabajo vida": [
            "equilibrar",
            "enfoco",
            "descanso",
            "me gust"
        ],
        "desorganizacion y dificultad": [
            "Aun no se gestionar",
            "Aún no tengo definido",
            "mal",
            "No lo hago",
            "como puedo"
        ],
        "tecnicas y metodos": [
            "técnica"
        ],
        "flexibilidad": [
            "epende"
        ]
    }
]

# 6 – Uso de Tecnología y Redes Sociales.xlsx
question_type[6] = [
    # Marca temporal
    "datetime",
    # Número de cuenta
    "account_number",
    # ¿Cuál es tu edad?
    "quantity",
    # ¿En qué semestre te encuentras?
    "quantity",
    # ¿Qué dispositivos electrónicos utilizas con mayor frecuencia?\n(Ej. teléfono móvil, tablet, computadora, etc.)
    {
        "telefono": ["teléfono", "móvil", "smartphone", "teléfono móvil"],
        "laptop": ["laptop", "computadora", "portátil"],
        "tablet": ["tablet"],
        "smartwatch": ["smartwatch"]
    },
    # ¿Cuántas horas al día usas dispositivos electrónicos para fines personales?
    "quantity",
    # ¿Utilizas dispositivos electrónicos durante las horas escolares? ¿Para qué actividades?
    mapping_yes_no,
    # ¿Qué redes sociales usas regularmente?\n(Ej: Instagram, Facebook, TikTok, Twitter, etc.)
    {
        "instagram": ["instagram", "ig"],
        "facebook": ["facebook", "fb"],
        "tikTok": ["tiktok", "tik tok", "tiktok"],
        "twitter": ["twitter", "x"],
        "whatsApp": ["whatsapp"],
        "snapchat": ["snapchat"],
        "reddit": ["reddit"],
        "youTube": ["youtube"],
        "telegram": ["telegram"]
    },
    # ¿Cuál es el principal motivo por el que utilizas las redes sociales?
    {
        "entretenimiento": ["entretenimiento", "ocio", "divertirme", "distraído", "ver videos", "memes"],
        "comunicación": ["comunicación", "comunicarme", "mensajería", "contacto con mis amigos", "estar al tanto de mis conocidos"],
        "distracción": ["distracción", "perder el tiempo", "pos nomás"],
        "aprender": ["aprender"],
        "contenido": ["contenido"]
    },
    # ¿En qué momentos del día sueles acceder a las redes sociales?
    {
        "mañana": ["mañana", "por la mañana"],
        "tarde": ["tarde", "por las tardes", "en la tarde"],
        "noche": ["noche", "noches", "por la noche", "en la noche"],
        "todo el día": ["todo el día", "siempre"],
        "horas libres": ["horas libres", "cuando estoy aburrido", "cuando tengo tiempo muertos", "en plazos del día"],
        "transporte": ["transporte", "ida y regreso"]
    },
    # ¿Utilizas las redes sociales mientras realizas tareas escolares o estudios? ¿Cómo influye en tu concentración?
    mapping_yes_no,
    # ¿Consideras que el uso de redes sociales afecta tu rendimiento académico? Explica tu respuesta.
    mapping_yes_no,
    # ¿Qué tipo de redes sociales prefieres y por qué?
    # ¿Has notado cambios en tu capacidad de concentración o productividad al usar redes sociales?
    mapping_yes_no,
    # ¿Crees que el uso excesivo de tecnología afecta la interacción con tus compañeros en el entorno escolar? Explica tu percepción.
    mapping_yes_no,
    # ¿Estás al tanto de las políticas o normas sobre el uso de tecnología en tu escuela? ¿Las consideras suficientes?
    mapping_yes_no,
    # ¿Recibes información o formación sobre cómo proteger tu privacidad y seguridad en línea? ¿Qué aspectos te gustaría reforzar?
    mapping_yes_no,
    # ¿Has sido testigo o víctima de ciberacoso o situaciones inseguras en redes sociales?
    mapping_yes_no,
    # ¿Te sientes preparado/a para hacer un uso responsable y seguro de la tecnología? ¿Por qué?
    mapping_yes_no,
    # ¿Qué tipo de talleres o formación adicional te gustaría recibir en relación con el uso de tecnología y redes sociales?
    {
        "uso de redes y tecnología": ["uso óptimo de las redes", "tener un buen uso de estas", "uso de tecnologías", "configurar un router"],
        "ciberseguridad": ["ciberseguridad", "seguridad", "seguridad y extracción de datos"],
        "formación técnica": ["técnicos", "curso IBM"],
        "situaciones de vulnerabilidad": ["reglamento de la UNAM", "cómo reaccionar ante una situación de vulnerabilidad"],
        "ninguna": ["ninguno", "ninguna", "no lo sé", "no estaría interesada", "nada"]
    },
    # ¿Cómo crees que las redes sociales influyen en tu estado de ánimo o bienestar emocional?
    mapping_raiz_debido_a_materias,
    # ¿Consideras que el uso de redes sociales ha mejorado o dificultado la comunicación y relación con tus compañeros? Detalla tu experiencia
    mapping_yes_no,
    # ¿Crees que el uso excesivo de tecnología puede contribuir al aislamiento o la desconexión social? Explica tu perspectiva
    mapping_yes_no,
    # ¿Qué medidas o estrategias propondrías para promover un uso saludable de la tecnología en la escuela?
    {
        "control del tiempo": ["medir el tiempo", "control del tiempo", "horarios establecidos",
                            "limitar para que se usa el internet", "medir el tiempo, tomar en cuenta lo utilidad de lo que se hace"],
        "educación y concientización": ["campañas", "más información", "hacer conciencia", "infografías", "talleres"],
        "acción individual y responsabilidad": ["responsabilidad de cada persona", "cada quien es responsable de lo que hace o no"],
        "uso académico del internet": ["el uso del internet esté limitado solo para lo académico", "no abusar del internet para buscar respuestas"],
        "condiciones del entorno": ["propiciar la convivencia", "no con amigos o en clases", "actividad para sustituir tiempo en tecnología"],
        "ninguna": ["ninguna", "no sé me ocurre ahora mismo", "no lo sé", "ninguno"]
    },
    # ¿Qué actividades o programas te gustaría que se implementaran para mejorar la educación digital y el manejo de redes sociales?
    {
        "educación y concientización": ["campañas sobre el tema", "saber el daño que podemos a hacer a otros",
                                        "seminarios", "infografías", "clases didácticas de redes responsables", "platicas", "eventos"],
        "talleres y formación Práctica": ["talleres de como implementar la tecnología", "talleres de comunicación",
                                        "talleres en la escuelas", "algún taller o actividad que nos enseñe el uso correcto", "talleres"],
        "clases en línea": ["clases en linea", "clases didácticas"],
        "sin propuesta": ["no sé me ocurre", "ninguno", "no tengo una en mente por ahora", "no tengo idea",
                        "creo que no es necesario", "no se", "ninguna"],
        "otros": ["algún evento"]
    }
]


SyntaxError: invalid syntax. Perhaps you forgot a comma? (4245287463.py, line 223)

## Proceso de Clasificación

In [418]:
# Lista de DataFrames donde se almacenarán los datos clasificados
questionnaires_classified = []
for questionnaire in questionnaires_clean:
  # Mismo cuestionario con mismas columnas
  questionnaires_classified.append(questionnaire.iloc[0:0])

In [419]:
for questionnaire_i, questionnaire in enumerate(questionnaires_clean):
    for question_i in range(questionnaire.shape[1]):
        current_type = question_type[questionnaire_i][question_i]
        column_data = questionnaire.iloc[:, question_i]

        if isinstance(current_type, str):
            classified_column = column_data.tolist()
        # Si el tipo no es un string, es un mapping
        else:
            classified_column = [classify_answer(str(answer), current_type) for answer in column_data]

        questionnaires_classified[questionnaire_i].iloc[:, question_i] = classified_column

In [420]:
questionnaires[1]

,Marca temporal,Número de cuenta,1. ¿Cuánto dinero generas al mes?,2. ¿Cuánto gastas mensualmente considerando todos tus gastos?,3. ¿Cuánto dinero logras ahorrar al mes?,"4. ¿Tienes dinero reservado para emergencias? Si sí, ¿cuánto aproximadamente?",5. ¿Tu dinero tiene liquidez o está comprometido en inversiones o deudas?,6. ¿Cuentas con inversiones a corto plazo?,7. ¿Cuentas con inversiones a largo plazo?,8. ¿Tienes un plan personal de retiro o planeas tener uno en el futuro?,9. ¿Cuántas tarjetas de crédito tienes?,10. ¿Qué porcentaje de tu línea de crédito tienes disponible actualmente?
0,2025-02-24 19:55:55.274,314687144,22,"Entre 15,000 - 16,000","Entre 2,000 - 5,000",5,"la gran mayoría si tiene liquidez, pero tambié...",no,no,si,2,0.95
1,2025-02-25 15:27:17.467,422075345,"$20,000","$10,000 aprox","$10,000","$40,000",Tiene liquidez,si,no,Tal vez lo considere en el futuro,1,0.85
2,2025-02-26 03:44:59.363,318104263,No genero.,4000,3000,"Si, 20000",Estoy comprometido en inversiones.,No,Si,Si,Ninguna,No aplica
3,2025-02-27 19:38:21.613,421039692,4000,3000,1000,2000,No,No,No,No,0,0
4,2025-02-27 20:07:15.769,317075261,19000,$10000,$4000,Si,Si,No,Si,Si,2,0.5
5,2025-02-28 00:22:20.116,422068044,0,3500,0,8000,no,si,no,no,0,0
6,2025-02-28 17:05:40.993,42107105,8000,5200,0,3000,tiene liquidez,no,no,no,2,8400
7,2025-02-28 17:24:39.661,422079312,0,8000,1000,"sí, unos 5k",inversiones y deudas de la tarjeta de crédito,"si, en mercado pago y nu",no,crear uno en el futuro aunque es más prometedo...,1,0.3
8,2025-02-28 17:25:08.189,319289619,"7,000 pesos",2.5,4.5,4000,Liquidez por ahora,Solo en Cripto monedas de mercado libre,Por el momento no,Planeo tener uno en el futuro,1,14
9,2025-02-28 17:47:44.002,317237748,4000,3500,300,"Si, unos 1000",Liquidez,Si,No,Planeo tener uno,1,0.2


In [421]:
questionnaires_classified[1]

,Marca temporal,Número de cuenta,1. ¿Cuánto dinero generas al mes?,2. ¿Cuánto gastas mensualmente considerando todos tus gastos?,3. ¿Cuánto dinero logras ahorrar al mes?,"4. ¿Tienes dinero reservado para emergencias? Si sí, ¿cuánto aproximadamente?",5. ¿Tu dinero tiene liquidez o está comprometido en inversiones o deudas?,6. ¿Cuentas con inversiones a corto plazo?,7. ¿Cuentas con inversiones a largo plazo?,8. ¿Tienes un plan personal de retiro o planeas tener uno en el futuro?,9. ¿Cuántas tarjetas de crédito tienes?,10. ¿Qué porcentaje de tu línea de crédito tienes disponible actualmente?
0,2025-02-24 19:55:55.274,314687144,22,"entre 15,000 16,000","entre 2,000 5,000",5,liquidez,no,no,sí,2,0.95
1,2025-02-25 15:27:17.467,422075345,"20,000","10,000 aprox","10,000","40,000",liquidez,sí,no,sí,1,0.85
2,2025-02-26 03:44:59.363,318104263,no genero.,4000,3000,"si, 20000",liquidez,no,sí,sí,ninguna,no aplica
3,2025-02-27 19:38:21.613,421039692,4000,3000,1000,2000,desconocido,no,no,no,0,0
4,2025-02-27 20:07:15.769,317075261,19000,10000,4000,si,liquidez,no,sí,sí,2,0.5
5,2025-02-28 00:22:20.116,422068044,0,3500,0,8000,desconocido,sí,no,no,0,0
6,2025-02-28 17:05:40.993,42107105,8000,5200,0,3000,liquidez,no,no,no,2,8400
7,2025-02-28 17:24:39.661,422079312,0,8000,1000,"s, unos 5k",liquidez,sí,no,no,1,0.3
8,2025-02-28 17:25:08.189,319289619,"7,000 pesos",2.5,4.5,4000,liquidez,None,no,no,1,14
9,2025-02-28 17:47:44.002,317237748,4000,3500,300,"si, unos 1000",liquidez,sí,no,no,1,0.2


# Procesamiento

## Funciones para procesar números

In [422]:
def process_numeric_input(value):
    """
    Procesa entradas numéricas (flotantes):
    - Valores únicos (quita comas, $ u otro formato)
    - Miles representados como "K" (e.g., 2.5K -> 2500)
    - Rangos (e.g., "2000 - 5000" o "entre 2000 y 5000" -> promedio del rango)
    - Limpia palabras como "aproximadamente" o similares
    - Procesa "no" y "nada" como 0
    """
    if pd.isna(value) or not isinstance(value, str):
        return None

    value = value.strip().lower()

    # Procesar "no" y "nada" como 0
    if value in ["no", "nada"]:
        return 0

    # Eliminar palabras irrelevantes como "aproximadamente"
    value = value.replace("$", "").replace("aproximadamente", "").replace("aprox", "")
    value = re.sub(r"[^\d.,\s\-k]", "", value)  # Remove non-numeric characters except ., - and k

    # Miles como K
    if "k" in value:
        try:
            return float(value.replace("k", "").replace(",", "").strip()) * 1000
        except ValueError:
            return None

    # Rangos con "entre"
    if "entre" in value:
        try:
            # Extraer todos los números en el texto
            range_values = [float(v.replace(",", "")) for v in re.findall(r"\d+(?:,\d+)?(?:\.\d+)?", value)]
            if len(range_values) == 2:  # Si hay exactamente dos números
                return sum(range_values) / len(range_values)  # Retornar el promedio
        except ValueError:
            return None

    # Rangos con "-"
    if "-" in value:
        try:
            range_values = [float(v.strip().replace(",", "")) for v in value.split("-")]
            return sum(range_values) / len(range_values)
        except ValueError:
            return None

    # Rangos con espacio (e.g., "entre 15,000 16,000")
    if " " in value:
        try:
            range_values = [float(v.replace(",", "")) for v in value.split() if v.replace(",", "").isdigit()]
            if len(range_values) == 2:
                return sum(range_values) / len(range_values)
        except ValueError:
            return None

    # Valores únicos
    try:
        return float(value.replace(",", ""))
    except ValueError:
        return None

def process_integer_input(value):
    """
    Procesa entradas enteras:
    - Quita comas, espacios y otros caracteres.
    - Convierte strings con enteros a enteros.
    - Convierte "no" o "nada" a 0
    """
    if pd.isna(value) or not isinstance(value, str):
        return None

    value = value.strip().lower()

    # "no" o "nada" a 0
    if value in ["no", "nada", "ninguno", "ninguna"]:
        return 0

    # Quita caracteres superfluos
    value = re.sub(r"[^\d-]", "", value)

    # Convierte a entero
    try:
        return int(value)
    except ValueError:
        return None

def process_percentage_input(value):
    """
    Procesa entradas de porcentajes:
    - Convierte valores como 0.95 a 95% (asume que valores menores a 1 son proporciones).
    - Convierte valores como "14" a 14% (asume que valores mayores o iguales a 1 son porcentajes).
    - Quita caracteres irrelevantes como "%".
    - Convierte "no" o "nada" a 0%.
    """
    if pd.isna(value) or not isinstance(value, str):
        return None

    value = value.strip().lower()

    # "no" o "nada" a 0%
    if value in ["no", "nada"]:
        return 0
    # "todo" a 100%
    elif value in ["todo", "toda"]:
        return 100

    # Quita caracteres irrelevantes como "%"
    value = value.replace("%", "").strip()

    # Convierte a flotante
    try:
        numeric_value = float(value)
        # Si el valor es menor a 1, asume que es una proporción (e.g., 0.95 -> 95%)
        if numeric_value < 1:
            return numeric_value * 100
        # Si el valor es mayor o igual a 1, asume que ya es un porcentaje
        return numeric_value
    except ValueError:
        return None

In [423]:
# Lista de DataFrames donde se almacenarán los datos clasificados
questionnaires_processed = []
for questionnaire in questionnaires_clean:
  # Mismo cuestionario con mismas columnas
  questionnaires_processed.append(questionnaire.iloc[0:0])

In [424]:
for questionnaire_i, questionnaire in enumerate(questionnaires_classified):
    for question_i in range(questionnaire.shape[1]):
        current_type = question_type[questionnaire_i][question_i]
        column_data = questionnaire.iloc[:, question_i]

        if current_type == "numeric":
            classified_column = column_data.apply(process_numeric_input)
        elif current_type == "integer":
            classified_column = column_data.apply(process_integer_input)
        elif current_type == "percentage":
            classified_column = column_data.apply(process_percentage_input)
        else:
            classified_column = [answer for answer in column_data]

        questionnaires_processed[questionnaire_i].iloc[:, question_i] = classified_column

In [425]:
questionnaires_processed[1]

,Marca temporal,Número de cuenta,1. ¿Cuánto dinero generas al mes?,2. ¿Cuánto gastas mensualmente considerando todos tus gastos?,3. ¿Cuánto dinero logras ahorrar al mes?,"4. ¿Tienes dinero reservado para emergencias? Si sí, ¿cuánto aproximadamente?",5. ¿Tu dinero tiene liquidez o está comprometido en inversiones o deudas?,6. ¿Cuentas con inversiones a corto plazo?,7. ¿Cuentas con inversiones a largo plazo?,8. ¿Tienes un plan personal de retiro o planeas tener uno en el futuro?,9. ¿Cuántas tarjetas de crédito tienes?,10. ¿Qué porcentaje de tu línea de crédito tienes disponible actualmente?
0,2025-02-24 19:55:55.274,314687144,22.0,15500.0,3500.0,5.0,liquidez,no,no,sí,2,95.0
1,2025-02-25 15:27:17.467,422075345,20000.0,10000.0,10000.0,40000.0,liquidez,sí,no,sí,1,85.0
2,2025-02-26 03:44:59.363,318104263,NaN,4000.0,3000.0,20000.0,liquidez,no,sí,sí,0,NaN
3,2025-02-27 19:38:21.613,421039692,4000.0,3000.0,1000.0,2000.0,desconocido,no,no,no,0,0.0
4,2025-02-27 20:07:15.769,317075261,19000.0,10000.0,4000.0,NaN,liquidez,no,sí,sí,2,50.0
5,2025-02-28 00:22:20.116,422068044,0.0,3500.0,0.0,8000.0,desconocido,sí,no,no,0,0.0
6,2025-02-28 17:05:40.993,42107105,8000.0,5200.0,0.0,3000.0,liquidez,no,no,no,2,8400.0
7,2025-02-28 17:24:39.661,422079312,0.0,8000.0,1000.0,5000.0,liquidez,sí,no,no,1,30.0
8,2025-02-28 17:25:08.189,319289619,7000.0,2.5,4.5,4000.0,liquidez,None,no,no,1,14.0
9,2025-02-28 17:47:44.002,317237748,4000.0,3500.0,300.0,1000.0,liquidez,sí,no,no,1,20.0


# Exportación a Datamart

## CSV (rápido)

In [426]:
output_folder = "export"
os.makedirs(output_folder, exist_ok=True)

for i, questionnaire in enumerate(questionnaires_processed):
    output_file = os.path.join(output_folder, f"questionnaire_{i + 1}.csv")
    questionnaire.to_csv(output_file, index=False, encoding="utf-8")
    print(f"Exported: {output_file}")

Exported: export\questionnaire_1.csv
Exported: export\questionnaire_2.csv
Exported: export\questionnaire_3.csv
Exported: export\questionnaire_4.csv
Exported: export\questionnaire_5.csv
Exported: export\questionnaire_6.csv
Exported: export\questionnaire_7.csv


# Auxiliar              

In [427]:
# Generar el código de todos los cuestionarios para question_types
output = ""
for i, questionnaire in enumerate(questionnaires):
  output += f"\question_type[{i}] = [\n"
  for column in questionnaire.columns:
    output += f"    # {column}\n\n"
  else:
    output += "]\n"

<>:4: SyntaxWarning: invalid escape sequence '\q'
<>:4: SyntaxWarning: invalid escape sequence '\q'
C:\Users\camargomau\AppData\Local\Temp\ipykernel_18436\2228946470.py:4: SyntaxWarning: invalid escape sequence '\q'
  output += f"\question_type[{i}] = [\n"
